In [11]:
from datasets import load_dataset
from collections import Counter
import numpy as np


In [12]:
aya = load_dataset(
    "CohereLabs/aya_dataset",
    "default",
    cache_dir="../data/raw/hf_cache"
)


Generating test split: 100%|██████████| 1750/1750 [00:00<00:00, 43294.06 examples/s]


In [13]:
# Dataset inspection:
print("whole dataset: ",aya)
print("Dataset keys: trian/test: ", aya.keys())
print("Features of each key: ", aya["train"].features)
print("Clean visualization of features: ", aya["train"].column_names)
print("Feature extraction: ", aya["train"]["inputs"])


whole dataset:  DatasetDict({
    train: Dataset({
        features: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'],
        num_rows: 202362
    })
    test: Dataset({
        features: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'],
        num_rows: 1750
    })
})
Dataset keys: trian/test:  dict_keys(['train', 'test'])
Features of each key:  {'inputs': Value('string'), 'targets': Value('string'), 'language': Value('string'), 'language_code': Value('string'), 'annotation_type': Value('string'), 'user_id': Value('string')}
Clean visualization of features:  ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']
Feature extraction:  Column(['Heestan waxaa qada Khalid Haref Ahmed \nOO ku Jiray Kooxdii Dur Dur!', 'Quels président des États-Unis ne s’est jamais marié ?', 'كم عدد الخلفاء الراشدين ؟ أجب على السؤال السابق.', 'கேள்வி: \nகீழ்க்காணும்  புதிருக்கான விடையை கண்டுபிடிக்கவும். \nகையில்லாமல் 

In [14]:
# studying one example
sample = aya["train"][11]
print(sample)
for key, value in sample.items():
    print(key,": ", value)


{'inputs': 'Scrivi una possibile continuazione di questo paragrafo - "L\'Italia ha partecipato ai XIV Giochi paralimpici estivi..."', 'targets': '"... di Londra, tenutisi tra il 29 agosto e il 9 settembre 2012, con una delegazione di 98 atleti concorrenti in 12 discipline. I portabandiera azzurri sono stati l\'arciere Oscar De Pellegrin durante la cerimonia d\'apertura e Alex Zanardi durante quella di chiusura. Gli azzurri si sono aggiudicati complessivamente 28 medaglie: 9 ori, 8 argenti e 11 bronzi."', 'language': 'Italian', 'language_code': 'ita', 'annotation_type': 're-annotations', 'user_id': '5dd2513170e3d607c99308d6b155214be6efb1c499378dfa7ba5aeef9ebbe525'}
inputs :  Scrivi una possibile continuazione di questo paragrafo - "L'Italia ha partecipato ai XIV Giochi paralimpici estivi..."
targets :  "... di Londra, tenutisi tra il 29 agosto e il 9 settembre 2012, con una delegazione di 98 atleti concorrenti in 12 discipline. I portabandiera azzurri sono stati l'arciere Oscar De Pelle

In [15]:
# Count the number of occurrencies for each of the desired languages:
training_languages = {"eng", "fra", "ita"}
control_languages = {"por", "spa"}
all_languages = {"eng", "fra", "ita", "por", "spa"}

aya_subset = aya["train"].filter(
    lambda x: x["language_code"] in all_languages
    )


Filter: 100%|██████████| 202362/202362 [00:01<00:00, 104156.83 examples/s]


In [16]:
# Counting the number of occurrencies for each language to understand the expected output that we can expect from the finetuning
print("Train counts:")
print(Counter(aya["train"]["language_code"]))
print(Counter(aya_subset["language_code"]))

print("\nTest counts:")
print(Counter(aya["test"]["language_code"]))


Train counts:
Counter({'plt': 14597, 'sin': 14524, 'tam': 14133, 'yor': 11758, 'zsm': 10073, 'por': 8997, 'vie': 8676, 'kir': 8622, 'tel': 8439, 'ary': 8090, 'som': 7704, 'pan': 6385, 'jpn': 6259, 'arb': 4995, 'zho': 4909, 'tur': 4046, 'npi': 4002, 'guj': 3989, 'eng': 3944, 'spa': 3854, 'mar': 3545, 'hau': 3512, 'wol': 2914, 'zul': 1833, 'mal': 1749, 'nld': 1733, 'pes': 1578, 'ben': 1534, 'ibo': 1534, 'pol': 1483, 'fra': 1422, 'sna': 1368, 'swe': 1310, 'gle': 1245, 'fil': 1241, 'amh': 1207, 'hin': 1153, 'pbt': 989, 'eus': 939, 'lit': 916, 'ind': 786, 'fin': 742, 'ita': 738, 'ceb': 727, 'tha': 724, 'nya': 688, 'urd': 654, 'ell': 623, 'arz': 529, 'ukr': 522, 'mya': 472, 'rus': 423, 'xho': 377, 'swh': 366, 'kor': 361, 'kan': 334, 'snd': 274, 'jav': 247, 'deu': 241, 'sun': 194, 'srp': 152, 'nso': 141, 'ars': 136, 'acq': 129, 'als': 120, 'hat': 106, 'hun': 98, 'dan': 97, 'ajp': 81, 'ckb': 79})
Counter({'por': 8997, 'eng': 3944, 'spa': 3854, 'fra': 1422, 'ita': 738})

Test counts:
Counter({'

In [17]:
training_data = aya_subset.filter(lambda x: x["language_code"] in training_languages)
comparison_data = aya_subset.filter(lambda x: x["language_code"] in control_languages)


Filter: 100%|██████████| 18955/18955 [00:00<00:00, 70216.42 examples/s]


In [18]:
train_splits = {}
val_splits = {}
test_splits = {}

for code in training_languages:
    language_data = training_data.filter(lambda x: x["language_code"] == code)

    split_80_20 = language_data.train_test_split(test_size=0.2, seed=17)
    train_splits[code] = split_80_20["train"]
    language_temp = split_80_20["test"]

    split_10_10 = language_temp.train_test_split(test_size=0.5, seed=11)
    val_splits[code] = split_10_10["train"]
    test_splits[code] = split_10_10["test"]


Filter: 100%|██████████| 6104/6104 [00:00<00:00, 67654.00 examples/s]


In [19]:
# Testing if what obtained makes sense
print(val_splits["ita"])
print(val_splits["fra"])
print(val_splits["eng"])


sample = val_splits["ita"][70]
print(sample)
for key, value in sample.items():
    print(key,": ", value)


Dataset({
    features: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'],
    num_rows: 74
})
Dataset({
    features: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'],
    num_rows: 142
})
Dataset({
    features: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id'],
    num_rows: 394
})
{'inputs': 'Scrivi una continuazione per questo testo: \n"Dal 1935 al 1940 ha giocato nel Brescia, con 29 presenze e 31 reti subite, di cui 1 in Serie A 1935-1936..."', 'targets': '"...e 28 in Serie B."', 'language': 'Italian', 'language_code': 'ita', 'annotation_type': 're-annotations', 'user_id': '5dd2513170e3d607c99308d6b155214be6efb1c499378dfa7ba5aeef9ebbe525'}
inputs :  Scrivi una continuazione per questo testo: 
"Dal 1935 al 1940 ha giocato nel Brescia, con 29 presenze e 31 reti subite, di cui 1 in Serie A 1935-1936..."
targets :  "...e 28 in Serie B."
language :  Italian
language_code :  ita
annotation_typ

In [20]:
from datasets import concatenate_datasets

train_ds = concatenate_datasets([train_splits[c] for c in training_languages]).shuffle(seed=11)
validation_ds = concatenate_datasets([val_splits[c] for c in training_languages]).shuffle(seed=12)
test_ds = concatenate_datasets([test_splits[c] for c in training_languages]).shuffle(seed=13)


In [21]:
spa_ds = comparison_data.filter(lambda x: x["language_code"] == "spa").shuffle(seed=42)
por_ds = comparison_data.filter(lambda x: x["language_code"] == "por").shuffle(seed=42)

control_spa = spa_ds.select(range(min(300, len(spa_ds))))
control_por = por_ds.select(range(min(300, len(por_ds))))


Filter: 100%|██████████| 12851/12851 [00:00<00:00, 69192.73 examples/s]


In [22]:
train_ds.save_to_disk("../data/processed/train")
validation_ds.save_to_disk("../data/processed/val")
test_ds.save_to_disk("../data/processed/test")
control_spa.save_to_disk("../data/processed/control_spa")
control_por.save_to_disk("../data/processed/control_por")


Saving the dataset (0/1 shards):   0%|          | 0/4882 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 49882.70 examples/s]
